# SRQ-FLY update optimization gate

Synthetic only: no dataset, checkpoint, test split, or accuracy tuning. Run the smoke gate first. Run the FLY-10000 timing cell only after the smoke gate passes. Timing is hardware-specific and does not replace the three-dataset held-out artifact.

In [ ]:
# Edit this cell only.
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'paper/srq-fly-draft'
WORK_DIR = '/content/SOHO-CL-update-opt'
OUTPUT_DIR = '/content/srq_fly_update_optimization'


In [ ]:
# Clone or fast-forward the declared branch without touching Google Drive.
import os, subprocess, sys
from pathlib import Path
repo = Path(WORK_DIR)
if not repo.exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
else:
    subprocess.run(['git','-C',WORK_DIR,'status','--short'], check=True)
    subprocess.run(['git','-C',WORK_DIR,'pull','--ff-only'], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'], check=True)
print('repo commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('torch:', __import__('torch').__version__, '| cuda:', __import__('torch').cuda.is_available())


In [ ]:
# Correctness tests: historical locked SRQ plus the opt-in implementation.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_math.py','tests/test_srq_fly_learner.py','tests/test_srq_fly_optimized.py','tests/test_srq_fly_heldout.py','tests/test_srq_fly_selfcontained.py'], check=True)


In [ ]:
# Small CUDA smoke gate. Expect two TASK lines and status=pass.
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
smoke_output = str(Path(OUTPUT_DIR) / 'smoke.json')
subprocess.run([sys.executable,'-u','tools/srq_fly_update_benchmark.py','--config','configs/srq_fly_update_optimization_smoke.json','--output',smoke_output,'--device','cuda'], check=True)


In [ ]:
# Dimension-matched FLY-10000 synthetic timing gate. It does not evaluate accuracy.
real_output = str(Path(OUTPUT_DIR) / 'fly10000.json')
subprocess.run([sys.executable,'-u','tools/srq_fly_update_benchmark.py','--config','configs/srq_fly_update_optimization_fly10000.json','--output',real_output,'--device','cuda'], check=True)


In [ ]:
# Show compact timing/state evidence and download it.
import json, zipfile
import pandas as pd
rows=[]
for name in ('smoke','fly10000'):
    payload=json.loads((Path(OUTPUT_DIR)/f'{name}.json').read_text())
    for method, values in payload['update_seconds'].items():
        rows.append({'scale':name,'method':method,'total_update_seconds':sum(values),'persistent_state_bytes':payload['persistent_state_bytes'][method]})
display(pd.DataFrame(rows))
archive='/content/srq_fly_update_optimization.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in Path(OUTPUT_DIR).glob('*.json'): z.write(path,arcname=path.name)
from google.colab import files
files.download(archive)
